In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import when, col, row_number, last
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import random

In [0]:


# -------------------------
# 1. Base identifiers
# -------------------------
order_ids = [f"O{i}" for i in range(1, 50001)]
customer_ids = [f"C{i}" for i in range(1, 50001)]

event_types = ["CREATED", "UPDATED", "COMPLETED", "CANCELLED"]
source_systems = ["web", "mobile", "api"]

base_time = datetime(2024, 1, 1, 9, 0, 0)

rows = []

# -------------------------
# 2. Generate dirty events
# -------------------------
for i in range(len(order_ids)):
    order_id = order_ids[i]
    customer_id = customer_ids[i]

    num_events = random.randint(1, 4)

    event_times = sorted(
        [base_time + timedelta(minutes=random.randint(0, 120)) for _ in range(num_events)]
    )

    for et in event_times:
        event_type = random.choice(event_types)

        order_amount = (
            None if random.random() < 0.15
            else round(random.uniform(100, 5000), 2)
        )

        ingestion_time = et + timedelta(minutes=random.randint(-3, 5))

        rows.append((
            order_id,
            customer_id,
            event_type,
            et,
            order_amount,
            random.choice(source_systems),
            ingestion_time
        ))

        # Introduce exact duplicate events
        if random.random() < 0.1:
            rows.append((
                order_id,
                customer_id,
                event_type,
                et,
                order_amount,
                random.choice(source_systems),
                ingestion_time
            ))

# -------------------------
# 3. Schema
# -------------------------
schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("event_type", StringType(), False),
    StructField("event_time", TimestampType(), False),
    StructField("order_amount", DoubleType(), True),
    StructField("source_system", StringType(), False),
    StructField("ingestion_time", TimestampType(), False)
])

# -------------------------
# 4. Create DataFrame
# -------------------------
orders_df = spark.createDataFrame(rows, schema)

display(orders_df.limit(10))


Write PySpark code to produce ONE FINAL RECORD PER order_id with the following rules:

✅ Business Rules

Deduplicate events

Exact duplicates may exist

Keep only one

Pick the latest valid state per order

Latest = by event_time

If event_time ties, use ingestion_time

Event precedence

If an order is CANCELLED, it should be the final state even if COMPLETED arrived later

Priority order:

CANCELLED > COMPLETED > UPDATED > CREATED


Order amount handling

Use the latest non-null order_amount

Even if the latest event has NULL

Output columns

order_id
customer_id
final_status
final_order_amount
final_event_time

In [0]:
display(orders_df.limit(10))

In [0]:
# What I wrote
window = Window.partitionBy("order_id") \
                .orderBy(col("event_time").asc(), col("ingestion_time").desc()) \
                .rowsBetween(Window.unboundedPreceding, Window.currentRow)

window1 = Window.partitionBy("order_id") \
                .orderBy(col("Rank_Event").asc(), col("event_time").desc(), col("ingestion_time").desc())

r_orders = orders_df.alias("o").withColumn("Rank_Event", when(col("event_type") == "CANCELLED", 1)
                                                        .when(col("event_type") == "COMPLETED", 2)
                                                        .when(col("event_type") == "UPDATED", 3)
                                                        .when(col("event_type") == "CREATED", 4)
                                                        ) \
                                .withColumn("final_order_amount", F.last("order_amount", ignorenulls = True).over(window)) \
                                .withColumn("Matching_Rank", F.rank().over(window1)) \
                                .filter(col("Matching_Rank") == 1) \
                                .select(col("order_id"), col("customer_id"), col("event_type").alias("final_status"), col("final_order_amount"), col("event_time").alias("final_event_time"))
display(r_orders)

In [0]:
# Correct Answer
dedup_df = orders_df.dropDuplicates([
    "order_id",
    "customer_id",
    "event_type",
    "event_time",
    "order_amount",
    "ingestion_time"
])



status_df = dedup_df.withColumn(
    "event_rank",
    when(col("event_type") == "CANCELLED", 1)
    .when(col("event_type") == "COMPLETED", 2)
    .when(col("event_type") == "UPDATED", 3)
    .otherwise(4)
)

status_window = Window.partitionBy("order_id") \
    .orderBy(col("event_rank"), col("event_time").desc(), col("ingestion_time").desc())

final_status_df = status_df.withColumn(
    "rn", row_number().over(status_window)
).filter(col("rn") == 1)

amount_window = Window.partitionBy("order_id") \
    .orderBy(col("event_time").desc(), col("ingestion_time").desc()) \
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

final_amount_df = status_df.withColumn(
    "final_order_amount",
    last("order_amount", ignorenulls=True).over(amount_window)
).select("order_id", "final_order_amount").distinct()

final_df = final_status_df.join(
    final_amount_df,
    on="order_id",
    how="left"
).select(
    "order_id",
    "customer_id",
    col("event_type").alias("final_status"),
    "final_order_amount",
    col("event_time").alias("final_event_time")
)

display(final_df)


Produce ONE ROW PER order_id with the following logic:

✅ Business Rules

Pick the latest SUCCESS payment per order

Latest = by payment_time

Tie-breaker = ingestion_time

If no SUCCESS exists

Pick the latest non-SUCCESS payment

(FAILED or PENDING)

Orders with no payments

Must still appear in output

Payment fields = NULL

Output schema

order_id
customer_id
order_amount
final_payment_status
final_payment_amount
final_payment_time


In [0]:
# -------------------------
# Orders data
# -------------------------
order_rows = []
base_time = datetime(2024, 1, 1, 9, 0, 0)

for i in range(1, 20001):
    order_rows.append((
        f"O{i}",
        f"C{i}",
        base_time + timedelta(minutes=random.randint(0, 60)),
        round(random.uniform(100, 5000), 2)
    ))

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_time", TimestampType(), False),
    StructField("order_amount", DoubleType(), False)
])

orders_df = spark.createDataFrame(order_rows, orders_schema)


In [0]:
payment_statuses = ["FAILED", "PENDING", "SUCCESS"]

payment_rows = []

for i in range(1, 20001):
    order_id = f"O{i}"

    # Skew: some orders have many payments
    num_payments = random.choice([1, 2, 3, 5, 50])

    for j in range(num_payments):
        payment_time = base_time + timedelta(minutes=random.randint(0, 120))
        ingestion_time = payment_time + timedelta(minutes=random.randint(-5, 10))

        status = random.choices(
            payment_statuses,
            weights=[0.4, 0.3, 0.3],
            k=1
        )[0]

        payment_rows.append((
            f"P{i}_{j}",
            order_id,
            status,
            round(random.uniform(100, 5000), 2),
            payment_time,
            ingestion_time
        ))

        # Duplicate payment event
        if random.random() < 0.1:
            payment_rows.append((
                f"P{i}_{j}",
                order_id,
                status,
                round(random.uniform(100, 5000), 2),
                payment_time,
                ingestion_time
            ))

payments_schema = StructType([
    StructField("payment_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("payment_status", StringType(), False),
    StructField("payment_amount", DoubleType(), False),
    StructField("payment_time", TimestampType(), False),
    StructField("ingestion_time", TimestampType(), False)
])

payments_df = spark.createDataFrame(payment_rows, payments_schema)


In [0]:
display(payments_df)
display(orders_df)

In [0]:
ranked_window = Window.partitionBy("order_id") \
                    .orderBy(col("status_rank"), col("payment_time").desc(), col("ingestion_time").desc())

payment_ranked_df = payments_df.withColumn("status_rank", when(col("payment_status") == "SUCCESS", 1)
                                                            .otherwise(2) ) \
                                .withColumn("row_ranked", row_number().over(ranked_window)) \
                                .filter(col("row_ranked") == 1)

final_df = orders_df.alias("o") \
            .join(payment_ranked_df.alias("p"), col("o.order_id") == col("p.order_id"), "left") \
            .select(
                col("o.order_id"),
                col("o.customer_id"),
                col("o.order_amount"),
                col("p.payment_status").alias("final_payment_status"),
                col("p.payment_amount").alias("final_payment_amount"),
                col("p.payment_time").alias("final_payment_time")
            )

display(final_df)
#order_id 
# customer_id
# order_amount
# final_payment_status
# final_payment_amount
# final_payment_time